<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.1-stationary-heat/Ex08.1_04_compare_and_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.1 · Notebook 04 — Compare, and Report

**Paired with L8.1 · Stationary Heat Transfer**

One equation, solved twice: once where the answer was known and once where the
geometry was real. This notebook puts the numbers side by side and assembles
the report.

What is marked is not whether your numbers match anyone else's. It is whether
you can say what you measured, what it means, and where it stops being true.

---

## 0 · Setup and what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.1-stationary-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
man = np.load(os.path.join("Ex08.1_outputs", "nb01_manufactured.npz"))
hole = np.load(os.path.join("Ex08.1_outputs", "nb02_hole.npz"))
flux = np.load(os.path.join("Ex08.1_outputs", "nb03_flux.npz"))

print("manufactured verification:")
print(f"    rel_L2 = {float(man['rel_L2']):.3e}")
print(f"   max_abs = {float(man['max_abs']):.3e}")
print(f"  boundary = {float(man['boundary']):.3e}")
print()
print("plate with hole: peak", f"{float(hole['peak']):.4f}",
      "at", tuple(np.round(hole["loc"], 3)))
print()
print(error_table(
    [[f"{w:g}", f"{p:.3e}", f"{f:.3e}"]
     for w, p, f in zip(flux["weights"], flux["pde"], flux["flux"])],
    ["w", "PDE residual", "flux term"]))
print()
print(f"flux balance: out {float(flux['out']):.4f}"
      f"   generated {float(flux['generated']):.4f}"
      f"   mismatch {float(flux['mismatch']):.2%}")

## 0b · Your personal seed

Every notebook in this set fixes the seed to 88 so the printed "what you should
see" blocks are true on any machine. That is right for checking your work and
wrong for reporting it — with one seed the whole cohort produces identical
numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints where the questions ask for it. Your supervisor can
regenerate exactly these numbers from your study number alone.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own collocation draw on the plate, and what the geometry implies for it.
your_pts = pb.sample_plate_with_hole(1500, seed=SEED)
phi = pb.hole_multiplier(your_pts).ravel()

print()
print(f"  your interior points    : {your_pts.shape[0]}")
print(f"  smallest multiplier     : {phi.min():.5f}   (must be > 0)")
print(f"  mean distance from hole : {phi.mean():.5f}   in level-set units")
print(f"  generated heat, Q/k = 10: {10.0 * pb.plate_area():.5f}")
print(f"  hole perimeter          : {pb.hole_perimeter():.5f}")

## 1 · Report questions

Each question names the slide that answers it. Replace every string below;
keep to the word limits, which are tight on purpose.

In [ ]:
# TODO: write your report. Every string below must be replaced.

Q1_SCALING = """
(120 words) **Why must the problem be scaled before training?** *(slide 8)*
Use the silicon-device numbers from the deck -- L around 1e-3 m, k around 150
W/mK, Q around 1e8 W/m^3 -- and say what a tanh network with Xavier
initialisation does with them unscaled. Then name the second reason scaling is
not optional in this exercise, which has to do with the multiplier.
"""

Q2_CONVECTION = """
(120 words) **Why can a convection condition not be hard-enforced?**
*(slides 6, 10)* State what a Robin condition constrains that Dirichlet and
Neumann do not, and why no multiplier of the kind you wrote in notebook 02 can
satisfy it. Say what follows for the loss.
"""

Q3_PURE_NEUMANN = """
(150 words) **What breaks if every face carries a flux condition?** *(slide 7)*
Give the degeneracy and the compatibility condition. Then say why the plate in
notebook 02 does not suffer from it, naming the specific thing that fixes the
temperature level -- and what you would have had to add if it were absent.
"""

Q4_FLUX_LEARNED = """
(150 words) **How do you know the flux condition was actually learned?**
*(slides 11, 22)* Quote your per-term losses from notebook 03 and your flux
balance mismatch. Say which of the two is the stronger evidence and why. If
your weight sweep did not behave as slide 11 predicts, say so -- that is a
finding, not an error.
"""

Q5_FEM = """
(150 words) **When would you use FEM instead, and why?** *(slide 24)* Be
specific about this problem: one steady solve, meshable geometry, known
conductivity. Then name the change to the problem statement that would flip
your answer, and say what property of the PINN it exploits.
"""

Q6_WHAT_YOU_DISTRUST = """
(120 words) Name the result in this exercise you trust least, and say exactly
what experiment would settle it. An answer naming a specific number and a
specific test scores higher than a general statement about needing more data.
"""

NAME = "your name"
GROUP = "your group"

raise NotImplementedError("Write your report, then delete this line")

## 2 · Check, assemble, save

In [ ]:
answers = {
    "1 · Scaling before training": (Q1_SCALING, 120),
    "2 · Why convection stays soft": (Q2_CONVECTION, 120),
    "3 · The pure-Neumann trap": (Q3_PURE_NEUMANN, 150),
    "4 · Evidence the flux condition was learned": (Q4_FLUX_LEARNED, 150),
    "5 · PINN or FEM": (Q5_FEM, 150),
    "6 · What you distrust": (Q6_WHAT_YOU_DISTRUST, 120),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or f"({limit} words)" in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = ["# Ex_08.1 — Stationary heat: a plate with a cooling hole", "",
             f"**{NAME}** · {GROUP}", "",
             f"study number {STUDENT_NUMBER} · seed {SEED}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "---", "",
             "## Results", "",
             f"- manufactured verification: relative L2 "
             f"{float(man['rel_L2']):.3e}, max abs {float(man['max_abs']):.3e}",
             f"- plate with hole: peak {float(hole['peak']):.4f} at "
             f"{tuple(np.round(hole['loc'], 3))}",
             f"- flux balance: out {float(flux['out']):.4f}, generated "
             f"{float(flux['generated']):.4f}, mismatch "
             f"{float(flux['mismatch']):.2%}", "",
             "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    report = "\n".join(lines)
    out_path = os.path.join("Ex08.1_outputs", "Ex08.1_report.md")
    with open(out_path, "w", encoding="utf-8") as fh:
        fh.write(report)
    print("wrote", out_path)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08.1_report.md into Ex08.1_report.pdf, with any figure
# saved as Ex08.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex08.1_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex08.1_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex08.1_report.pdf")
print("written Ex08.1_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex08.1_report.pdf")
except ImportError:
    pass


## 3 · Extensions

- Replace the insulated outer edges with convection, $-k\,\partial T/\partial n
  = h(T - T_\infty)$. Compute the Biot number first and predict what you will see.
- Move the hole 20% closer to one edge. How does the peak move?
- Make `Q_OVER_K` unknown and recover it from five interior "sensor" values.
- Give the plate two materials with different `k` and observe the interface.

---

## 4 · What Ex_08.1 was for

The same elliptic operator as Ex_07.1, carrying physical meaning and sitting on
a domain that is not a rectangle.

* **Geometry is the work.** The multiplier, the normals and the samplers took
  more effort than the network did, and none of them involved training.
* **A level set does three jobs at once** — it says whether a point is in the
  domain, it hard-enforces a value on the curved boundary, and its normalised
  gradient is the outward normal.
* **Sample a curved boundary by arc length**, because the curvature, and the
  flux with it, is not spread evenly around it.
* **A conservation check beats an error norm.** The flux balance can be
  computed without knowing the answer, which is exactly the situation you will
  be in on every problem worth solving.

Next: **Ex_08.2**, the same plate and the same hole, with time.